# exp068 — compact con el trainer público de R7 (heliosli `compact_train.py` v9.8) y sus flags reproducibles, fold 0

Padre exp066_compact_f0. Única variable: el trainer y los flags de la receta R7 (`heliosli/rsna-knee-compact-r7cnxfsss`,
manifest `run_config`, `compact_train.py` sha256 04df7e2e…) que no necesitan datos privados: EMA 0.999, slice jitter 0.25,
mirror 0.5, aug v2 (+grid fp32), muestreo cover (stride 3), draw-seed, syn-decomp. Mismos datos, etiquetas, caché y split
que exp066. Decide la pseudo L2 de la compact sola en fold 0 contra exp066 (ver `SPEC.md`).

El código de heliosli NO se copia a git (licencia "no new open-source license"): se lee de
`/workspace/data/public_assets/heliosli__rsna-knee-compact-r7cnxfsss/src/` (fuentes embebidas en el kernel público
`heliosli/0-957-latest-dataset-score-in-one-click`, sha256 = manifest R7) y el port (`compact_train_port.py`)
se escribe en el disco local del pod. La caché px384_v4 la construye `build_px_cache.py` (esta carpeta) con el camino de
inferencia de heliosli (`compact_infer.CStudies`).

Fases (`PHASE`): `smoke` (1 época, 48 estudios + caché de evaluación chica) · `train` (16 épocas) · `eval` (predicción fold 0 +
gold, métricas, blend). Celdas: 0 parámetros · 1 setup · 2 datos, split y targets · 3 port del modelo · 4 entrenamiento ·
5 evaluación · 6 guardar · 7 verdict.

In [ ]:
EXP = "exp068_compact_r7_receta"
PADRE = "exp066_compact_f0"
FOLDS = [0]
SEED = 0                               # seed 0 de la receta (manifest: E_lrprior_s0_*)
SPLIT = "split_v1_seed42"
SELECT_ON = "pseudo"                   # no hay selección: modelo de la última época (como la receta)
SMOKE = True
PHASE = "smoke"                        # smoke | train | eval
PX_TRAIN = "/root/px384_v4_train"      # caché folds 1-4 (disco local del pod)
PX_EVAL = "/root/px384_v4_eval"        # caché fold 0 + gold (se construye tras borrar la de train)
RUN_DIR = "/root/run_exp068"           # salida de compact_train (disco local); el .pt se copia a results/
PA = "/workspace/data/public_assets"
CODE_DIR = "/root/compact/code"        # cloud_code de heliosli (copia local)
SRC_DIR = "/root/compact/src_r7"       # fuentes R7 de heliosli (copia local) + compact_train_port.py
INIT = "/root/compact/pretrained/convnext_small_in22k_ft_in1k.safetensors"   # timm/convnext_small.fb_in22k_ft_in1k
LABELS_SOFT = "/workspace/data/labels/public/dreaddevelopment/rsna-knee-labels/labels_llm_soft.parquet"
L2_CSV = "/workspace/data/labels/public/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv"
L3_CSV = "/workspace/data/labels/public/pilkwang/rsna-knee-llm-labels/report_labels_v2.csv"
L4_CSV = "/workspace/data/labels/public/lixin73/rsna-knee-llm-report-labels-sol56/labels_llm_gpt56sol.csv"
V1_CSV = "/workspace/data/labels/report_labels_v1.csv"
REF_DIR = "/workspace/users/rodrigo/wt_exp045_raptor_aug_espacial/results/exp045_raptor_aug_espacial"
REF_PSEUDO_SWA = 0.8894654941741685    # exp045 metrics.json swa_f0.pseudo_auc
REF_GOLD_SWA = 0.9201083029612395      # exp045 metrics.json swa_f0.gold_auc
REF_PSEUDO_COMPACT = 0.8755489889579056  # exp066 metrics.json per_fold.0.pseudo_auc (padre; decide)
REF_GOLD_COMPACT = 0.897173253306936   # exp066 metrics.json per_fold.0.gold_auc (padre; veto)
# exp068: flags de la receta R7 (manifest run_config) que no necesitan datos privados
R7_FLAGS = ["--slice-jitter", "0.25", "--ema", "0.999", "--syn-decomp", "--mirror", "0.5", "--aug", "v2", "--aug-grid-fp32",
            "--draw-seed", "--train-sampling", "cover", "--cover-stride", "3"]
L2_THRESHOLD = 0.5
P_CLIP = 0.01                          # prior consenso recortado a [0.01, 0.99] antes de la razón de verosimilitud
# receta (manifest run_config de heliosli/rsna-knee-compact-v1)
ARM = "E_lrprior"
BACKBONE = "convnext_small"
EPOCHS = 16
BATCH = 2
TRAIN_WINDOWS = 176
CROP = 352
LR = 1e-4
HEAD_LR = 1e-3
WD = 0.05
SLICES = 3
PHYS_MM = 3.5
EVAL_STRIDE = 1                        # manifest eval_stride (inferencia del blend-mine)
SPATIAL_ATTN = True                    # los 4 modelos públicos traen la atención espacial
COMPILE = True                         # R7 run_config compile true (misma matemática)
WORKERS = 12
SMOKE_N = 48
NOTES = "exp066 con el trainer R7 v9.8 + flags públicos (EMA, jitter, mirror, aug v2, cover, syn-decomp); sin estados/extent/fm-aux/pesos de celda"

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import torch

HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(REPO))
from src.knee.config import LABELS, current_user
from src.knee.experiments import git_info, guardar_metricas
from src.knee.metrics import mean_auc
from src.knee.splits import load_split

assert SELECT_ON == "pseudo"
assert (REPO / "experiments" / EXP / "SPEC.md").exists(), "Primero escribe la SPEC"
assert PHASE in ("smoke", "train", "eval")
if SMOKE:
    assert PHASE == "smoke"
RESULTS = REPO / "results" / EXP
RESULTS.mkdir(parents=True, exist_ok=True)
GIT = git_info(REPO)
AUTHOR = current_user()
# fuentes públicas de heliosli: sha256 = manifest src_sha256 (compact-v1)
SRC_SHA = {"compact_train.py": "04df7e2e154b2c2521696908f272940c63ce17e9b6337b85847fee64504eeaba",
           "px_cache.py": "b13e38a7ad3bd496f8027e02209653e224b0926db4f0853eeb0d076a2d6e2632",
           "knee_anchor_bank_v2.py": "e564807a129dfcd3c984348917436ee6083066a4d685340587ebd1d3ea14f309",
           "side_common.py": "bbcc5d027834e6a6fd48c4a8cf228e25661b7522c792587c840fe4d7cd5c0566"}
for n, h in SRC_SHA.items():
    assert hashlib.sha256((Path(SRC_DIR) / n).read_bytes()).hexdigest() == h, n
    assert hashlib.sha256((Path(PA) / "heliosli__rsna-knee-compact-r7cnxfsss" / "src" / n).read_bytes()).hexdigest() == h, n
print(f"{EXP} · {AUTHOR} · git {GIT['commit']} ({GIT['branch']}) · PHASE {PHASE} · torch {torch.__version__}")

In [ ]:
split = load_split(SPLIT)
def _read_soft(path, name):
    df = pd.read_parquet(path) if str(path).endswith(".parquet") else pd.read_csv(path, low_memory=False)
    df["StudyInstanceUID"] = df.StudyInstanceUID.astype(str)
    assert not df.StudyInstanceUID.duplicated().any(), name
    df = df.set_index("StudyInstanceUID")[LABELS].apply(pd.to_numeric, errors="coerce").astype(float)
    assert df.stack().between(0.0, 1.0).all(), name
    return df

# etiquetas de entrenamiento = consenso de exp031/exp045 (media por celda de Raptor, L2, L3, L4; media de las presentes)
SOURCES = {"raptor": _read_soft(LABELS_SOFT, "raptor"), "L2": _read_soft(L2_CSV, "L2"),
           "L3": _read_soft(L3_CSV, "L3"), "L4": _read_soft(L4_CSV, "L4")}
_all = sorted(set().union(*[s.index for s in SOURCES.values()]))
_stack = np.stack([s.reindex(_all)[LABELS].to_numpy(dtype=float) for s in SOURCES.values()])
_n = np.isfinite(_stack).sum(0)
soft = pd.DataFrame(np.where(_n > 0, np.nansum(_stack, 0) / np.maximum(_n, 1), np.nan),
                    index=pd.Index(_all, name="StudyInstanceUID"), columns=LABELS)
l2 = _read_soft(L2_CSV, "L2")
pseudo_bin = (l2 > L2_THRESHOLD).astype(float).where(l2.notna())
v1 = pd.read_csv(V1_CSV); v1["StudyInstanceUID"] = v1.StudyInstanceUID.astype(str)
v1 = v1.set_index("StudyInstanceUID")[LABELS].astype(float); v1_bin = v1.where(v1.isin([0.0, 1.0]))
train_csv = pd.read_csv("/workspace/data/raw/train.csv"); train_csv["StudyInstanceUID"] = train_csv.StudyInstanceUID.astype(str)
gold = train_csv.dropna(subset=LABELS, how="all").set_index("StudyInstanceUID")[LABELS].astype(float)
gold_ids = list(gold.index)
assert len(gold) == 58 and set(gold_ids) == set(split.loc[split.is_gold == 1, "StudyInstanceUID"])
tr_ids = split.loc[split.fold >= 1, "StudyInstanceUID"].tolist()
va_ids = split.loc[split.fold == 0, "StudyInstanceUID"].tolist()
assert not set(tr_ids) & set(gold_ids) and not set(tr_ids) & set(va_ids)

# targets en el formato de compact_train: L|prior (probabilidad) y L|lr_prior (razón de verosimilitud), solo folds 1-4.
# lr = odds(p) / odds(base_c), base_c = media de p en train  ->  el óptimo de lr_loss con sesgo inicial logit(base_c) es p.
P = soft.reindex(tr_ids)[LABELS]
base = P.mean(0).clip(1e-3, 1 - 1e-3)
Pf = P.fillna(base).clip(P_CLIP, 1 - P_CLIP)          # celda sin fuente -> base (lr = 1, sin información)
odds = lambda x: x / (1 - x)
LR_ = odds(Pf) / odds(base.clip(P_CLIP, 1 - P_CLIP))
tg = pd.DataFrame({"StudyInstanceUID": tr_ids, "is_gold": "0", "review_holdout": "0",
                   "fold": split.set_index("StudyInstanceUID").loc[tr_ids, "fold"].astype(int).astype(str).values})
for L in LABELS:
    tg[L + "|prior"] = Pf[L].round(6).values
    tg[L + "|lr_prior"] = LR_[L].round(6).values
    tg[L + "|strict"] = ""
gtg = pd.DataFrame({"StudyInstanceUID": gold_ids, "is_gold": "1", "review_holdout": "0", "fold": "-1"})
for L in LABELS:
    gtg[L + "|prior"] = ""; gtg[L + "|lr_prior"] = ""; gtg[L + "|strict"] = ""
targets = pd.concat([tg, gtg], ignore_index=True)
TARGETS = Path(RUN_DIR) / "targets_exp068.csv"
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
if PHASE == "smoke":
    idx = json.load(open(Path(PX_TRAIN) / "index.json"))
    have = [u for u in tr_ids if u in idx["done"]][:SMOKE_N]
    targets = targets[targets.StudyInstanceUID.isin(have) | (targets.is_gold == "1")]
    TARGETS = Path(RUN_DIR) / "targets_exp068_smoke.csv"
targets.to_csv(TARGETS, index=False)
print(f"train {len(tr_ids)} · val {len(va_ids)} · gold {len(gold_ids)} · targets {TARGETS.name} filas {len(targets)}")
print("base (media consenso train):", dict(zip(LABELS, base.round(3).tolist())))
print("celdas sin fuente en train:", int(P.isna().sum().sum()), "· lr min/max", float(LR_.min().min()), float(LR_.max().max()))

In [ ]:
# Port de compact_train.py (fuente pública intacta + UN cambio): los folds salen de la columna `fold` de los targets
# (split_v1_seed42) en vez del hash sha1 % K.  Con --folds 5 --only-folds 0 el modelo "fold 0" entrena con todo el pool
# de la caché de train (folds 1-4); fold 0 y gold no están en esa caché, así que compact_train no predice nada al final:
# la predicción la hace la celda 5 con la caché de evaluación y las mismas funciones del trainer.
src = (Path(SRC_DIR) / "compact_train.py").read_text(encoding="utf-8")
before = 'pool_fold = np.array([int(hashlib.sha1(u.encode()).hexdigest()[:8], 16) % K for u in pool])'
after = 'pool_fold = np.array([int(by_uid[u]["fold"]) for u in pool])   # exp068: split_v1_seed42 folds (port)'
assert src.count(before) == 1
PORT = Path(SRC_DIR) / "compact_train_port.py"
PORT.write_text(src.replace(before, after), encoding="utf-8")
PORT_SHA = hashlib.sha256(PORT.read_bytes()).hexdigest()
print("port", PORT, PORT_SHA[:16])

In [ ]:
def train_args(out, epochs):
    a = [sys.executable, "-u", str(PORT), "--px", PX_TRAIN, "--targets", str(TARGETS), "--out", out,
         "--arms", ARM, "--backbone", BACKBONE, "--init", INIT, "--seeds", "1", "--folds", "5", "--only-folds", "0",
         "--epochs", str(epochs), "--batch", str(BATCH), "--train-windows", str(TRAIN_WINDOWS), "--crop", str(CROP),
         "--lr", str(LR), "--head-lr", str(HEAD_LR), "--wd", str(WD), "--slices", str(SLICES), "--phys-mm", str(PHYS_MM),
         "--eval-stride", str(EVAL_STRIDE), "--grad-ckpt", "--workers", str(WORKERS), "--ckpt-every", "1",
         "--resume", "--i-have-user-authorisation-to-finetune"]
    if SPATIAL_ATTN:
        a.append("--spatial-attn")
    if COMPILE:
        a.append("--compile")
    return a + R7_FLAGS

OUT = str(Path(RUN_DIR) / ("smoke" if PHASE == "smoke" else "f0"))
MODEL_PT = Path(OUT) / "models" / f"{ARM}_s0_f0.pt"
TRAIN_S = None
if PHASE in ("smoke", "train"):
    if PHASE == "smoke":
        shutil.rmtree(OUT, ignore_errors=True)
    env = dict(os.environ, PYTHONPATH=f"/root/pylib:{SRC_DIR}:{CODE_DIR}", PYTHONDONTWRITEBYTECODE="1")
    t0 = time.time()
    with open(RESULTS / f"train_{PHASE}.log", "a") as fh:
        p = subprocess.Popen(train_args(OUT, 1 if PHASE == "smoke" else EPOCHS), stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, env=env, cwd=SRC_DIR)
        for line in p.stdout:
            fh.write(line); fh.flush()
            if "epoch" in line or "pool" in line or "init" in line or "Error" in line or "trainable" in line:
                print(line.rstrip(), flush=True)
        rc = p.wait()
    TRAIN_S = time.time() - t0
    assert rc == 0, f"compact_train rc={rc} (ver results/{EXP}/train_{PHASE}.log)"
    assert MODEL_PT.exists()
    json.dump({"train_seconds": TRAIN_S, "port_sha256": PORT_SHA}, open(Path(OUT) / "exp068_train_info.json", "w"))
    if PHASE == "train":
        shutil.copy(MODEL_PT, RESULTS / f"{ARM}_s0_f0.pt")     # pesos en el volumen (nunca a git)
        shutil.copy(Path(OUT) / "exp068_train_info.json", RESULTS / "train_info.json")
    print(f"entrenamiento {PHASE}: {TRAIN_S / 3600:.2f} h")

In [ ]:
# Predicción con las funciones del propio trainer: Studies(mode "eval", eval stride 1, phys 3.5), collate, recorte
# central 352, bf16, chunk 512 (= compact_train.predict).  Luego métricas y el blend rank-mean con exp045 SWA fold 0.
metrics_eval = None
if PHASE in ("smoke", "eval"):
    for p_ in ("/root/pylib", SRC_DIR, CODE_DIR):
        if p_ not in sys.path:
            sys.path.insert(0, p_)
    import importlib
    CT = importlib.import_module("compact_train_port")
    dev = torch.device("cuda")
    px_eval = PX_EVAL if PHASE == "eval" else PX_EVAL + "_smoke"
    eidx = json.load(open(Path(px_eval) / "index.json"))
    if PHASE == "eval" and not MODEL_PT.exists():
        MODEL_PT = RESULTS / f"{ARM}_s0_f0.pt"
    model = CT.Net(BACKBONE, None, crop=CROP, state_spec=None, in_chans=SLICES, spatial=SPATIAL_ATTN, extent=False,
                   syn_decomp=True)
    model.load_state_dict(torch.load(MODEL_PT, map_location="cpu"))
    model = model.to(dev).to(memory_format=torch.channels_last).eval()

    @torch.no_grad()
    def predict(uids):
        uids = [u for u in uids if u in eidx["done"]]
        ds = CT.Studies(px_eval, uids, TRAIN_WINDOWS, "eval", 0, eval_stride=EVAL_STRIDE, slices=SLICES, phys_mm=PHYS_MM)
        dl = torch.utils.data.DataLoader(ds, batch_size=1, shuffle=False, num_workers=4, collate_fn=CT.collate)
        out = np.zeros((len(uids), len(LABELS)), np.float32)
        o = (eidx["config"]["size"] - CROP) // 2
        for img, meta, mask, idx in dl:
            with torch.autocast("cuda", dtype=torch.bfloat16):
                z = model(img[..., o:o + CROP, o:o + CROP].to(dev), meta.to(dev), mask.to(dev), chunk=512)
            out[idx.numpy()] = torch.sigmoid(z.float()).cpu().numpy()
        return pd.DataFrame(out, index=pd.Index(uids, name="StudyInstanceUID"), columns=LABELS)

    t0 = time.time()
    pv, pg = predict(va_ids), predict(gold_ids)
    EVAL_S = time.time() - t0
    miss_v, miss_g = len(set(va_ids) - set(pv.index)), len(set(gold_ids) - set(pg.index))
    pv.add_prefix("pred_").reset_index().to_csv(RESULTS / f"oof_f0{'_smoke' if SMOKE else ''}.csv", index=False)
    pg.add_prefix("pred_").reset_index().to_csv(RESULTS / f"gold_f0{'_smoke' if SMOKE else ''}.csv", index=False)

    def score(pred, ref):
        ids = [u for u in pred.index if u in ref.index]
        return mean_auc(ref.loc[ids, LABELS].to_numpy(), pred.loc[ids, LABELS].to_numpy())

    def load_ref(name):
        d = pd.read_csv(Path(REF_DIR) / name); d["StudyInstanceUID"] = d.StudyInstanceUID.astype(str)
        d = d.set_index("StudyInstanceUID"); d.columns = [c[5:] for c in d.columns]
        return d[LABELS]

    def rank_blend(a, b, w=0.5):
        ids = [u for u in a.index if u in b.index]
        return (w * a.loc[ids].rank(pct=True) + (1 - w) * b.loc[ids].rank(pct=True))

    rv, rg = load_ref("oof_swa_f0.csv"), load_ref("gold_swa_f0.csv")
    ref_p, ref_g = score(rv, pseudo_bin), score(rg, gold)
    bv, bg = rank_blend(rv, pv), rank_blend(rg, pg)
    m = {"compact": {"pseudo": score(pv, pseudo_bin), "v1": score(pv, v1_bin), "gold": score(pg, gold)},
         "exp045_swa_recomputed": {"pseudo": ref_p, "gold": ref_g},
         "blend_rank_0.5_0.5": {"pseudo": score(bv, pseudo_bin), "v1": score(bv, v1_bin), "gold": score(bg, gold),
                                "n_val": len(bv), "n_gold": len(bg)},
         "missing": {"val": miss_v, "gold": miss_g}, "eval_seconds": EVAL_S}
    # criterios de la SPEC (contra los números de exp045 metrics.json; el recálculo es solo control)
    dp = m["blend_rank_0.5_0.5"]["pseudo"]["mean"] - REF_PSEUDO_SWA
    dg = m["blend_rank_0.5_0.5"]["gold"]["mean"] - REF_GOLD_SWA
    m["delta_blend_vs_exp045"] = {"pseudo": dp, "gold": dg}
    # exp068 decide contra el padre exp066 (compact sola); el blend con exp045 se reporta
    dpc = m["compact"]["pseudo"]["mean"] - REF_PSEUDO_COMPACT
    dgc = m["compact"]["gold"]["mean"] - REF_GOLD_COMPACT
    m["delta_compact_vs_exp066"] = {"pseudo": dpc, "gold": dgc}
    m["decision_auto"] = "killed" if dgc < -0.02 else ("worked" if dpc >= 0.005 else "didnt")
    metrics_eval = m
    print(f"compact f0: pseudo {m['compact']['pseudo']['mean']:.4f} (Δ vs exp066 {dpc:+.4f}) · v1 {m['compact']['v1']['mean']:.4f} · "
          f"gold {m['compact']['gold']['mean']:.4f} (Δ {dgc:+.4f}, veto) → {m['decision_auto']}")
    print(f"exp045 SWA recalculado: pseudo {ref_p['mean']:.4f} gold {ref_g['mean']:.4f} (metrics.json {REF_PSEUDO_SWA:.4f} / {REF_GOLD_SWA:.4f})")
    print(f"blend 0.5/0.5: pseudo {m['blend_rank_0.5_0.5']['pseudo']['mean']:.4f} (Δ {dp:+.4f}) · "
          f"gold {m['blend_rank_0.5_0.5']['gold']['mean']:.4f} (Δ {dg:+.4f}) (informativo) · faltan val {miss_v} gold {miss_g}")

In [ ]:
if PHASE in ("smoke", "eval"):
    info = {}
    for f_ in (Path(OUT) / "exp068_train_info.json", RESULTS / "train_info.json"):
        if f_.exists():
            info = json.load(open(f_)); break
    cache_meta = {}
    for name, d in (("train", PX_TRAIN), ("eval", PX_EVAL)):
        f_ = Path(d) / "index.json"
        if f_.exists():
            ix = json.load(open(f_))
            cache_meta[name] = {"studies": len(ix["done"]), "errors": len(ix["errors"]), "rows": ix["cursor"],
                                "sec_per_study_mean": float(np.mean(list(ix.get("seconds", {0: 0}).values())))}
    metrics = {"schema_version": 1, "exp": EXP, "author": AUTHOR, "parent": PADRE, "split": SPLIT,
               "split_sha256": split.attrs["sha256"], "git": GIT, "timestamp_utc": datetime.now(timezone.utc).isoformat(),
               "phase": "smoke" if SMOKE else "screen", "select_on": "none (last epoch, recipe)",
               "params": {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, list))},
               "recipe": {"source": "heliosli/rsna-knee-compact-r7cnxfsss manifest run_config (flags públicos)", "compact_train_sha256": SRC_SHA["compact_train.py"],
                          "port_sha256": info.get("port_sha256"), "deviations": [
                              "targets: consenso exp031 (Raptor+L2+L3+L4) como L|prior y L|lr_prior = odds(p)/odds(base); los targets opusv11b no son públicos",
                              "sin --state-targets/--pair-rank/--rare-boost/--extent: necesitan los estados opusv11b (no públicos); sin pesos de celda plr7 (no públicos); sin --fm-aux (fastMRI+, datos externos); sin review holdout; caché sin sidefix; --grad-ckpt por memoria (misma matemática)",
                              "folds: split_v1_seed42 (train folds 1-4), no hash sha1 % 4",
                              "caché px384_v4 reconstruida con el camino de inferencia público (compact_infer.CStudies) desde DICOM"]},
               "per_fold": {"0": {"n_train": len(tr_ids), "n_val": len(va_ids),
                                  "pseudo_auc": metrics_eval["compact"]["pseudo"]["mean"],
                                  "gold_auc": metrics_eval["compact"]["gold"]["mean"],
                                  "v1_auc": metrics_eval["compact"]["v1"]["mean"],
                                  "pseudo_per_label": metrics_eval["compact"]["pseudo"]["per_label"],
                                  "gold_per_label": metrics_eval["compact"]["gold"]["per_label"]}},
               "pseudo_auc": metrics_eval["compact"]["pseudo"]["mean"], "gold_auc": metrics_eval["compact"]["gold"]["mean"],
               "blend": metrics_eval, "decision_auto": metrics_eval["decision_auto"],
               "cache": cache_meta,
               "gpu_hours_train": (info.get("train_seconds") or 0) / 3600,
               "gpu_hours_total": ((info.get("train_seconds") or 0) + metrics_eval["eval_seconds"]) / 3600}
    path = guardar_metricas(RESULTS, metrics)
    if SMOKE:
        os.replace(path, RESULTS / "metrics_smoke.json"); path = RESULTS / "metrics_smoke.json"
    print("guardado", path)

## 7. Verdict
Llena la sección 3 de `SPEC.md` con `results/exp068_compact_r7_receta/metrics.json` y la fila de `docs/EXPERIMENTOS.md`.
Criterios fijos en la SPEC: `worked` si pseudo compact − 0.8755 ≥ +0.005 y gold ≥ 0.8772.